# Step 4 — ESIM on `overture_case2s`

**Why:** the word2vec baseline (`01_word2vec.ipynb`) averages each address into one vector, so it cannot tell `MAIN ST` from `MAPLE ST` (case2s AUC ≈ 0.6). ESIM lets the two addresses look at each other's words before the comparison.

**Spec (2026-11-02):** frozen word2vec (200-d, skip-gram, 50 epochs) with learned `UNK` and masked `PAD` → BiLSTM (200) → soft alignment both ways (PAD masked with −∞) → `[a; ã; a−ã; a⊙ã]` → composition BiLSTM (200) → masked mean + max pooling → `[v_a; v_b]` → MLP (200, dropout 0.3) → match probability. Training protocol identical to Step 3 (BCE, Adam 1e-3, `is_better` with the case1 guardrail, patience 3, max 20 epochs). Symmetry: original `[v_a; v_b]`, swap test later.

**Build plan:** all layers first, one shape pass through the whole model, then three tests (PAD mask, softmax axis, tiny-set overfit).

## 0. Setup
**Why:** one place for imports and the constants every cell uses.

In [1]:
# Setup: imports and constants
%load_ext autoreload
%autoreload 2

import copy
import os
import re
import sys
import time
from collections import Counter
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset, Subset
from gensim.models import Word2Vec

from helpers.pairs import load_pairs
from helpers.check_model_performance import review_typical_slices, find_random_sample_given_mask
from siamese_model.data_plumbing import PairDataset   # word-agnostic: returns raw strings
from siamese_model.encoder_step import BiLSTM          # pack -> LSTM -> unpack, unit-agnostic
from siamese_model.training_step import train_epoch, predict   # same batch format as our collate
from siamese_model.real_training import evaluate, is_better, broken_run_problems   # Step 3's per-slice val row, early-stopping rule, health check
from esim.tokenizer import tokenize   # same tokenisation word2vec was trained with

DATASET = "overture_case2s"
EMB_DIM, HIDDEN, MLP_HIDDEN, DROPOUT = 200, 200, 200, 0.3
BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-3, 20, 3
THRESHOLD, CASE1_GUARDRAIL = 0.5, 98.0   # same as siamese_model/real_training.py
PAD_ID, UNK_ID = 0, 1

# word2vec setting: the one switch between the two ESIM runs (2026-11-05)
#   2 = original run (rare house numbers -> UNK), 1 = every word gets a vector
W2V_MIN_COUNT = 1
W2V_PATH = f"checkpoints/word2vec_50_mc{W2V_MIN_COUNT}.model"      # from 01_word2vec.ipynb
CHECKPOINT_DIR = f"checkpoints/esim_mc{W2V_MIN_COUNT}"            # runs + saved predictions for this setting (git-ignored)


pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

## 1. Load the pairs
**Why:** same pair files and shared schema as word2vec and Step 3.
**Does:** loads train / val / test. Test stays untouched until the final evaluation.
**Output:** `train`, `val`, `test` DataFrames and their label arrays

In [2]:
# Load: train / val / test pairs
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
print(len(train), len(val), len(test))

202568 41865 43844


## 2. The word2vec model
**Why:** ESIM's word vectors come from the 50-epoch word2vec model. `01_word2vec.ipynb` never saved its model, and a retrained one differs run to run (Python's string hashing is randomised per kernel start), so it should be trained once and saved.
**Does:** gets the trained model into this notebook (gensim can save a model to a file and load it back; where the training and saving happen is your call).
**Output:** `w2v`, a trained gensim model with 200-d vectors

In [3]:
# word2vec model: train once and save / load
EMB_DIM = 200
w2v = Word2Vec.load(W2V_PATH)   # set in Setup
assert w2v.wv.vector_size == EMB_DIM
print(w2v.epochs, len(w2v.wv.key_to_index))

50 6150


## 3. Vocabulary and embedding matrix
- **Background**: Even though our `word2vec` model technically does the id-encoding as well as vector-encoding, we will split it up into two stage. `ID-Conversion` and `Vector-Conversion`. `Vector-Conversion` happens in this layer. `ID-Conversion` occurs in vocab
- Before we input anything, our `DataLoader` will upload all word_sequences as a series of **id's**, not words
- Then: in our embedding layer, we will take the embedding table from our trained `word2vec` model and place it into `nn.Embedding`. 
- **A Note on nn.Embeddings**: We often think of embeddings as being similar to a dict, but it's more like a list. If we give it a list of embeddings, the key will be the `index` and the value will be the embedding itself. So when we initially add two `zero-embeddings`, it essentially assign `0,1` to a zero embedding. 
- **What are we training on?**: We want our `word2vec` embeddings to be frozen, but we want to have a trainable `<unk>` parameter, because we any arbitrary embedding, will likely implicitly mean something on the embedding space if not trained. We get both by creating an `Embedding Layer` whose **weights are frozen**, which avoids updating gradients, while assigning `<unk>` using `Parameter` to allow only `<unk>` embeddings to be updated

##### Forward Dorward
- When we initially run our id through the embedding layer we get `out` of size $(n, m, h)$
- Our `is_unk` creates a mask of size $(n, m, 1)$, and essentially serves as a boolean mask
- `self.unk` is of size $(h)$, but when used will be broadcasted to size $(n,m,h)$
- when we use `torch.where`, 
    - `is_unk`, serves as the boolean, of which value should we use. 
    - `self.unk` is used as a constant value if `1`
    - `out` serves as a `mapping_function`
    - The reason `self.unk` and `out` work differently is because of their sizes (partially due to broadcasting), affect their output

In [4]:
# Vocab with PAD / UNK, embedding matrix
vocab = {"PAD": PAD_ID,"UNK": UNK_ID, **{k:v+2 for k,v in w2v.wv.key_to_index.items()}}

class EmbeddingLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = w2v
        matrix = torch.cat([torch.zeros(2, EMB_DIM), torch.tensor(w2v.wv.vectors)])
        self.frozen = nn.Embedding.from_pretrained(matrix, freeze = True)
        self.unk = nn.Parameter(torch.zeros(EMB_DIM))
        

    def forward(self, ids):
        out = self.frozen(ids)
        is_unk = (ids == UNK_ID).unsqueeze(-1)
        out = torch.where(is_unk, self.unk, out)
        return out

Embedding checks (written by Claude, 2026-11-04): each word gets its own word2vec row (the +2 shift), `PAD` gives zeros, `UNK` gives the learned `unk`. `unk` is set to ones first, because at init it equals the zero `PAD` row and the `UNK` check couldn't tell them apart.


In [5]:
# Embedding checks: word -> its own word2vec row, PAD -> zeros, UNK -> unk
emb = EmbeddingLayer()
with torch.no_grad():
    emb.unk.fill_(1.0)

n_words = len(w2v.wv.index_to_key)
words = [w2v.wv.index_to_key[i] for i in (0, 1, n_words // 2, n_words - 1)]   # first, second, middle, last
for w in words:
    assert torch.equal(emb(torch.tensor([[vocab[w]]]))[0, 0], torch.tensor(w2v.wv[w])), w
assert (emb(torch.tensor([[PAD_ID]])) == 0).all()
assert torch.equal(emb(torch.tensor([[UNK_ID]]))[0, 0], emb.unk)
print("embedding checks passed on", words)


embedding checks passed on ['ST', 'RD', 'SIMS', '1685']


## 4. Encode a batch
- We just tokenize our variables
- We have to minimize/limit size to avoid batch-size blowup (This more of concern when dealing with `chars`, but not so much for words)
- We pad it so they can be turned into a 2D tensor


In [6]:
# encode_batch (from siamese_model/encode_batch_step.py, now at word level)
MAX_BATCH_FEATURE_SIZE = 85
def encode_batch(strings, vocab, max_len = MAX_BATCH_FEATURE_SIZE):
    strings = [tokenize(s)[:max_len] for s in strings]
    batch_len = max([len(s) for s  in strings])
    ret = []
    for string in strings:
        res = [vocab.get(word, UNK_ID) for word in string]
        res += [PAD_ID] * (batch_len - len(res))
        ret.append(res)
    lengths = [len(s) for s  in strings]
    return torch.tensor(ret, dtype = torch.long), torch.tensor(lengths, dtype = torch.long)


- We were considering that an empty string could silently break gradients, test shows we have no empty addresses, so this is a non-issue

In [7]:
all_addr = pd.concat([d[c] for d in (train, val, test) for c in ("address_a", "address_b")])
empty = all_addr[all_addr.map(tokenize).str.len() == 0]
print(len(empty)); empty.value_counts().head()


0


Series([], Name: count, dtype: int64)

## 5. Dataset and DataLoaders
**Why:** feeds the model one batch at a time, encoding each batch only when it is drawn.
**Does:** a `Dataset` returning raw address strings and the label, a collate function that encodes both sides, and loaders (train shuffled, val and test not).
**Output:** `train_loader`, `val_loader`, `test_loader`

In [8]:
def make_collate(vocab):
    def collate(samples):
        # samples = [(address_a, address_b, label), ...] -> encode each side separately
        addr_a, addr_b, labels = zip(*samples)
        ids_a, len_a = encode_batch(list(addr_a), vocab)
        ids_b, len_b = encode_batch(list(addr_b), vocab)
        y = torch.tensor(labels, dtype=torch.float32)
        return ids_a, len_a, ids_b, len_b, y
    return collate

# Loaders (PairDataset imported from siamese_model/data_plumbing.py; make_collate stays here so it uses the word-level encode_batch)
train_dataset = PairDataset(train, train_label)
val_dataset = PairDataset(val, val_label)
test_dataset = PairDataset(test, test_label)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle = True, collate_fn = make_collate(vocab))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle = False, collate_fn= make_collate(vocab))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle = False, collate_fn= make_collate(vocab))

## 6. Encode: BiLSTM
- **BiLSTM**: its output_size is called `hidden_size` NOT `OUTPUT_SIZE` because the resultant sequence is just them dragging along the short-term memory at each timestep
- **dropout**: dropout is a little interesting and does not need to specify size, because it's not a layer w/ weights. It's merely a `mask`, whose size is equivelant to the **input_dims**
    - **But can't we infer input size for all things?**: We can, which is why we also have `nn.LazyLinear` which does exactly that. The ouput_size, however, cannot be inferred, and is a design choice.

In [9]:
# Context encoder: dropout -> shared BiLSTM (BiLSTM imported from siamese_model/encoder_step.py)
class ContextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.dropout = nn.Dropout(p=DROPOUT)
        self.bilstm = BiLSTM(input_size=EMB_DIM, hidden=EMB_DIM)

    def forward(self, vectors, lengths):
        dropout_val = self.dropout(vectors)
        BiLSTM_val = self.bilstm(dropout_val, lengths)
        return BiLSTM_val


## 7. Align: cross attention
- **What does @ do?**: `@` understands that for 3d operations with size (n, m_a, h) @ (n, h, m_b) that `n` is representative of number of rows, which means instead of 3D dot-product, we instead n operations of (m_a, b) x (h, m_b) 
    - This is also why we do `.transpose(1,2)`, which swaps `dim=1` and `dim=2`
    - The result shape is (B, m_a, m_b)
- **How do we create the Intial Mask?**: 
    - When we do B.size(1), we create w/ `torch.arange(B.size(1))`, our output_size is (m_b)
    - Next `len_b` is initially (n), but after `[:, None]` it becomes (n, 1)
    - Now we have to deal with `broadcasting`. After broadcasting:
        - `arange`: (m_b) -> (1, m_b) -> (n, m_b)
        - `len_b`: (n, 1) -> (n, m_b)
        - `result`: (n, m_b)
        - **NOTE**, we initially add 1 to right of `len_b`, cause broadcasting only attempts to add 1s to left
    - We can think of it like a `dict` where each tensor of length `seq_b_length` corresponds to a row in `rows`
- **How do we do Filled Mask?**
    - Firstly we handle mask whose shape is current (n, m_b) becomes (n, 1, m_b)
        - We use `~` to get the inverse because `original_mask` includes all values who are *within the length*
    - Now, we need to update mask to match the size of e. In order to understand, we can think of the initial mask representing a `dict`. That dict says for THIS ROW, we only use these versions of `seq_b`. Then, in order to apply it to the `attention_matrix`, we just need to think, now just apply it and tell all `seq_a` to ignore these values for `seq_b`
    - `softmax`takes the last dimension (which is `dim=2`), and makes it so their sum becomes 1

In [10]:
# Soft alignment with PAD mask
def attention_weights(A, B, len_B):
    # B (rows, seq_b_length, embedding)
    e = A @ B.transpose(1,2) # (rows, seq_a_length, seq_b_length)
    mask = torch.arange(B.size(1), device = B.device) < len_B[:, None] # (rows, seq_b_length)
    return e.masked_fill(~mask[:, None, :], float("-inf")).softmax(dim = -1) # (rows, seq_a_length, seq_b_length)

def align(A, B, len_B):
    w = attention_weights(A, B, len_B)
    return w @ B # (rows, seq_a_length, seq_b_length) x (rows, seq_b_length, embedding) -> (rows, seq_a_length, embedding)


A check to ensure that A_tilde produced by `align` gives the same shape as `original_A`

In [11]:
emb, enc = EmbeddingLayer(), ContextEncoder()
ids_a, len_a, ids_b, len_b, y = next(iter(train_loader))

A = enc(emb(ids_a), len_a)
B = enc(emb(ids_b), len_b)

A_tilde = align(A, B, len_b)
B_tilde = align(B, A, len_a)
assert A_tilde.shape == A.shape and B_tilde.shape == B.shape

We want to ensure that our attention `softmax` actually runs on the correct_dimension (dim = 2).
- This is important cause otherwise `softmax` can succeed while silently running on the wrong axis, messing up our `attention`/model

In [12]:
mask_index = 4
X, Y = torch.randn((1,8, 5)), torch.randn((1,8, 5))   # not A / B: those names hold the real batch used by later checks
len_Y = torch.tensor([mask_index])
res = attention_weights(X, Y, len_Y)
assert torch.allclose(res[:, :, :mask_index].sum(-1), torch.ones(res.shape[:2]), atol = 1e-6)
assert torch.allclose(res[:, :, mask_index:].sum(-1), torch.zeros(res.shape[:2]), atol = 1e-6)

## 8. Enhance
**Why:** puts match and mismatch signals side by side so later layers can read them.
**Does:** per word, concatenates `a`, `ã`, `a − ã` and `a ⊙ ã`.
**Output:** `M_a`, `M_b` (batch × length × 4H)

In [13]:
# Enhancement
def enhance(A, len_A, B, len_B):
    tilde_A = align(A, B, len_B)
    tilde_B = align(B, A, len_A)

    A_res = torch.cat([A, tilde_A, A - tilde_A, A * tilde_A], dim = 2)
    B_res = torch.cat([B, tilde_B, B - tilde_B, B * tilde_B], dim = 2)

    return A_res, B_res

1) Check that shape is same size except for embedding which should be 4 times larger
2) first 400 (2 * EMB_DIM) should be `original_A`
3) These tensors have `require_grad`, meaning they can be linked back to the model

In [14]:
M_a, M_b = enhance(A, len_a, B, len_b)
assert M_a.shape == (*A.shape[:2], 4 * A.size(-1))       # (n, m_a, 1600)
assert torch.equal(M_a[..., :400], A)                    # first block is A itself
assert M_a.requires_grad                                 # gradient history kept


## 9. Compose: second BiLSTM
**Why do we only process one side?**: We only process one side because later inside the `ESIM Module`, we will run `enhance` and run `EncodeAttention` per word_sequence then
- Add `relu` to avoid running a simple linear function

In [15]:
# Composition BiLSTM

class EncodeAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4*2*EMB_DIM, EMB_DIM)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(DROPOUT)
        self.BiLSTM = BiLSTM(EMB_DIM, EMB_DIM)

    def forward(self, vectors, lengths):
        linear_val = self.linear(vectors)
        relu_val = self.relu(linear_val)
        dropout_val = self.dropout(relu_val)
        BiLSTM_val = self.BiLSTM(dropout_val, lengths)
        return BiLSTM_val


- Check that the output embedding size matches twice the embedding dimension
- Make sure that BiLSTM padded comparisons are not computed and are `0`

In [16]:
comp = EncodeAttention()
C_a = comp(M_a, len_a)
C_b = comp(M_b, len_b)
assert C_a.shape == (*M_a.shape[:2], EMB_DIM * 2)
assert (C_a[0, len_a[0]:] == 0).all()

## 10. Pool
**Output Notes**: 
- (B, 800), since word_sequences get compressed
- concatenated `mean/max` into same embedding

In [17]:
# Masked mean + max pooling
class MeanMaxPooling(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, h, lengths):
        lengths = lengths.to(h.device)
        mask = torch.arange(h.size(1), device=h.device)[None, :] < lengths[:, None]   # (B, T), True = real position
        m = mask.unsqueeze(-1)  # (B, T, 1), broadcasts over features
        mean_pool = (h * m).sum(1) / m.sum(1) # pads add 0 and don't count: (B, T, D) -> (B, D)
        max_pool = h.masked_fill(~m, float("-inf")).max(dim=1).values
        return torch.cat([mean_pool, max_pool], dim = 1)

In [18]:
pooling = MeanMaxPooling()
v_a, v_b = pooling(C_a, len_a), pooling(C_b, len_b)

## 11. Head and the assembled model
**Why:** turns the pair into one match score.
**Does:** concatenates `[v_a; v_b]`, passes it through the MLP (hidden layer + dropout) to one logit, and wires sections 3–10 into a single `ESIM` model.
**Output:** `ESIM` model returning logits (batch,)

In [19]:
class ClassifierHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = nn.Linear(EMB_DIM * 4 * 2, EMB_DIM)
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(DROPOUT)
        self.linear2 = nn.Linear(EMB_DIM, 1)
        

    def forward(self, vectors):
        linear1_val = self.linear1(vectors)
        tanh_val = self.tanh(linear1_val)
        dropout_val = self.dropout(tanh_val)
        linear2_val = self.linear2(dropout_val)
        return linear2_val.squeeze(-1)

# ESIM model
class ESIM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = EmbeddingLayer()
        self.context_encoder = ContextEncoder()
        self.encode_attention = EncodeAttention()
        self.pooling = MeanMaxPooling()
        self.head = ClassifierHead()

    def forward(self, ids_a, len_a, ids_b, len_b):
        embedding_a = self.embedding(ids_a)
        embedding_b = self.embedding(ids_b)

        context_a = self.context_encoder(embedding_a, len_a)
        context_b = self.context_encoder(embedding_b, len_b)

        A_res, B_res = enhance(context_a, len_a, context_b, len_b)

        attention_a = self.encode_attention(A_res, len_a)
        attention_b = self.encode_attention(B_res, len_b)

        pooling_a = self.pooling(attention_a, len_a)
        pooling_b = self.pooling(attention_b, len_b)

        final_vector = torch.cat([pooling_a, pooling_b], dim = 1)        
        return self.head(final_vector)



Check that the classifier_head, generates a 1D tensor whose length equals number of rows

In [20]:
head = ClassifierHead()
x = torch.cat([v_a, v_b], dim = -1)
logits = head(x)
assert logits.shape == (x.size(0),)

In [21]:
esim = ESIM()
ids_a, len_a, ids_b, len_b, y = next(iter(train_loader))
res = esim(ids_a, len_a, ids_b, len_b)
assert res.shape == (ids_a.size(0),)

## 12. One pass and tests
**Why:** shape bugs crash, but a missing PAD mask or a wrong softmax axis only makes the numbers worse, so those need explicit checks.
**Does:** (a) one batch through the whole model, printing the shape after each stage; (b) mask test: the same pair padded to two different lengths gives the same output; (c) axis check: each row of the attention weights sums to 1, using addresses of different lengths; (d) overfit test: a tiny set reaches near-zero loss.
**Output:** passing asserts

#### B) Mask Test
Testing if our end-to-end code can properly filter out padding tokens
- make sure to set to `eval` to not use `dropout`. By default starts in `train` mode
- use `torch.allclose` to compare since dot-products will not return exact result

In [22]:
# Shape pass and tests

def modified_ids(ids, lengths):
    mask = torch.arange(ids.size(1)) >= lengths[:, None]   # position == length is the first PAD
    new_ids = torch.where(mask, 2, ids)
    return new_ids

# Testing Masking End-to-End
orig_ids = [[2,0,0], [2,2,0], [1,2,3]]
orig_lens = [1,2,3]

esim = ESIM().eval()
ids_a, len_a = torch.tensor(orig_ids), torch.tensor(orig_lens)
ids_b, len_b = torch.tensor([[3,4,5], [4,0,0], [5,3,0]]), torch.tensor([3,1,2])   # different lengths from A, so a len_a/len_b swap would fail
mod_a, mod_b = modified_ids(ids_a, len_a), modified_ids(ids_b, len_b)

assert torch.allclose(esim(ids_a, len_a, ids_b, len_b), esim(mod_a, len_a, mod_b, len_b), atol = 1e-6)


#### D) Overfit Test
- Unlike in normal training, our goal is to overfit the set
- This is so we're able to test the gradients of our model and assert the MODEL CAN LEARN

In [23]:
toy = pd.DataFrame([
    ("12 main st",        "12 main st",         1),   # identical
    ("12 main st",        "12 main street",     1),   # abbreviation
    ("45 oak avenue",     "45 oak ave",         1),   # abbreviation
    ("301 elm road",      "301 elm raod",       1),   # typo
    ("88 pine st apt 2",  "88 pine st",         1),   # extra unit (building-level match)
    ("12 main st",        "14 main st",         0),   # case1: same street, different number
    ("45 oak avenue",     "47 oak avenue",      0),   # case1
    ("301 elm road",      "301 elm street",     0),   # case2-style: same number, different street type
    ("12 main st",        "990 harbor view dr", 0),   # easy
    ("88 pine st",        "5 quarry lane",      0),   # easy
], columns=["address_a", "address_b", "label"])
toy_labels = toy["label"]

toy_ds = PairDataset(toy, toy_labels)
toy_loader = DataLoader(toy_ds, batch_size=len(toy_ds), shuffle=False, collate_fn=make_collate(vocab))   # one batch = the whole toy set

#TEST: Checks to make sure our dataset is not accidentally filled with words that work is <unk> on encoder
[w for s in pd.concat([toy.address_a, toy.address_b]) for w in tokenize(s) if w not in vocab]

['RAOD']

In [24]:
model = ESIM()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()
for epoch in range(300):
    loss = train_epoch(model, toy_loader, opt, loss_fn)
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d} loss {loss:.4f}")
assert loss < 0.05

epoch   0 loss 0.6920
epoch  50 loss 0.0001
epoch 100 loss 0.0001
epoch 150 loss 0.0001
epoch 200 loss 0.0002
epoch 250 loss 0.0001


#### E) Frozen vs learned embedding (written by Claude, 2026-11-04)
- One training step must leave the word2vec rows unchanged and move `unk`
- Needs `UNK` words in the batch, or `unk` gets no gradient: the toy set has `APT` and `RAOD`


In [25]:
# One training step: word2vec rows unchanged, unk moved
model = ESIM()
frozen_before = model.embedding.frozen.weight.clone()
unk_before = model.embedding.unk.detach().clone()
opt = torch.optim.Adam(model.parameters(), lr=LR)
train_epoch(model, toy_loader, opt, nn.BCEWithLogitsLoss())
assert torch.equal(model.embedding.frozen.weight, frozen_before), "word2vec rows changed: not frozen"
assert not torch.equal(model.embedding.unk.detach(), unk_before), "unk did not move: no gradient reached it"


## 13. Training
**Why:** same protocol as Step 3 so the numbers are comparable.
**Does:** per epoch, `train_epoch`, then evaluate on val per slice; best epoch by `is_better` (case1 guardrail 98%, then higher `found`, tie → lower val loss); stop after 3 epochs without improvement. Time one epoch first, then decide how many seeds.
**Output:** history, best epoch's row and weights, saved to a git-ignored checkpoint folder

In [26]:
SEED = 0
N_TIMING_BATCHES = 200

torch.manual_seed(SEED)
timing_model = ESIM()
timing_opt = torch.optim.Adam(timing_model.parameters(), lr=LR)
subset_loader = DataLoader(Subset(train_dataset, range(N_TIMING_BATCHES * BATCH_SIZE)), batch_size=BATCH_SIZE, collate_fn = make_collate(vocab))

# train/val loaders come from §2 (train shuffled); don't rebuild them here

t0 = time.perf_counter()
train_epoch(timing_model, subset_loader, timing_opt, nn.BCEWithLogitsLoss())
train_seconds = time.perf_counter() - t0
print(f"train: {train_seconds / N_TIMING_BATCHES:.3f} s/batch -> {train_seconds / N_TIMING_BATCHES * len(train_loader) / 60:.1f} min/epoch")

t0 = time.perf_counter()
predict(timing_model, val_loader)
print(f"val pass: {time.perf_counter() - t0:.1f} s")

train: 0.060 s/batch -> 3.1 min/epoch
val pass: 11.2 s


In [27]:
def run_training(seed=SEED):
    torch.manual_seed(seed)
    model = ESIM()
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr = LR)
    loss_fn = nn.BCEWithLogitsLoss()

    best, best_state, epochs_since_best = None, None, 0
    history = []
    for epoch in range(MAX_EPOCHS):
        train_loss = train_epoch(model, train_loader, opt, loss_fn)

        row = {"epoch": epoch, "train_loss": train_loss, **evaluate(model, val_loader, val, val_label)}
        print(row)
        history.append(row)

        if is_better(row, best):
            best, best_state, epochs_since_best = row, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_since_best += 1
            if epochs_since_best >= PATIENCE:
                break
    if best is None:                                      # no epoch reached the case1 guardrail -> keep the last epoch's weights
        best_state = copy.deepcopy(model.state_dict())
    return pd.DataFrame(history), best, best_state

In [28]:
# CHECKPOINT_DIR comes from Setup (one folder per W2V_MIN_COUNT); existing runs are skipped
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for seed in [0, 1, 2]:
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt")
    if os.path.exists(ckpt_path):
        continue

    history, best, best_state = run_training(seed = seed)

    history.to_csv(os.path.join(CHECKPOINT_DIR, f"history_seed_{seed}.csv"), index = False)
    torch.save({
        "state_dict": best_state, "vocab": vocab,
        "config": {"emb_dim": EMB_DIM, "hidden": HIDDEN, "mlp_hidden": MLP_HIDDEN, "dropout": DROPOUT,
                   "lr": LR, "max_len": MAX_BATCH_FEATURE_SIZE, "dataset": DATASET,
                   "w2v_min_count": W2V_MIN_COUNT},
        "seed": seed, "best": best}, ckpt_path)
    print(f"finishing seed:{seed} -> best: {best}")
    print(f"problems: {broken_run_problems(history)}")


{'epoch': 0, 'train_loss': 0.06516188401493657, 'val_loss': 0.06143098324537277, 'found': 77.79171894604768, 'case1': 98.08431898991365, 'easy': 99.97823152945274}
{'epoch': 1, 'train_loss': 0.03525187653536637, 'val_loss': 0.04241344705224037, 'found': 78.79548306148055, 'case1': 98.2729845439373, 'easy': 99.97387783534329}
{'epoch': 2, 'train_loss': 0.030876243606985807, 'val_loss': 0.0394878014922142, 'found': 77.03889585947303, 'case1': 98.92605761555765, 'easy': 99.99129261178109}
{'epoch': 3, 'train_loss': 0.028216755670409505, 'val_loss': 0.0441012978553772, 'found': 76.53701380175659, 'case1': 99.21631231405559, 'easy': 99.9825852235622}
{'epoch': 4, 'train_loss': 0.026676849867585256, 'val_loss': 0.043203987181186676, 'found': 74.90589711417817, 'case1': 98.83898120600827, 'easy': 99.9825852235622}
finishing seed:0 -> best: {'epoch': 1, 'train_loss': 0.03525187653536637, 'val_loss': 0.04241344705224037, 'found': 78.79548306148055, 'case1': 98.2729845439373, 'easy': 99.97387783

In [36]:
# TEMPORARY queue cell (2026-11-05): ESIM on fastText vectors, same protocol. Delete after the run;
# the fastText option moves into the Setup switch next session.
from gensim.models import FastText

w2v = FastText.load("checkpoints/fasttext_50_mc1.model")          # EmbeddingLayer reads the global w2v
assert w2v.wv.vector_size == EMB_DIM
vocab = {"PAD": PAD_ID, "UNK": UNK_ID, **{k: v + 2 for k, v in w2v.wv.key_to_index.items()}}
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=make_collate(vocab))
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(vocab))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(vocab))

CHECKPOINT_DIR = "checkpoints/esim_ft_mc1"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
for seed in [0, 1, 2]:
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt")
    if os.path.exists(ckpt_path):
        continue
    history, best, best_state = run_training(seed=seed)
    history.to_csv(os.path.join(CHECKPOINT_DIR, f"history_seed_{seed}.csv"), index=False)
    torch.save({"state_dict": best_state, "vocab": vocab,
                "config": {"emb_dim": EMB_DIM, "hidden": HIDDEN, "mlp_hidden": MLP_HIDDEN, "dropout": DROPOUT, "lr": LR,
                           "max_len": MAX_BATCH_FEATURE_SIZE, "dataset": DATASET, "w2v": "fasttext_50_mc1"},
                "seed": seed, "best": best}, ckpt_path)
    print(f"finishing seed:{seed} -> best: {best}")
    print(f"problems: {broken_run_problems(history)}")

    model = ESIM()                                            # save val/test predictions for 14d
    model.load_state_dict(best_state)
    for split, loader in [("val", val_loader), ("test", test_loader)]:
        preds = (predict(model, loader) >= THRESHOLD).int().numpy()
        np.save(os.path.join(CHECKPOINT_DIR, f"preds_{split}_seed_{seed}.npy"), preds)


{'epoch': 0, 'train_loss': 0.06211077274186125, 'val_loss': 0.05372440814971924, 'found': 78.92095357590966, 'case1': 98.0625498875263, 'easy': 99.94775567068658}
{'epoch': 1, 'train_loss': 0.03485945930802605, 'val_loss': 0.05212624371051788, 'found': 79.92471769134254, 'case1': 98.38183005587403, 'easy': 99.96517044712438}
{'epoch': 2, 'train_loss': 0.02961536655470271, 'val_loss': 0.043158188462257385, 'found': 79.17189460476789, 'case1': 99.1292359045062, 'easy': 99.97387783534329}
{'epoch': 3, 'train_loss': 0.027857066193671443, 'val_loss': 0.04758794605731964, 'found': 77.28983688833124, 'case1': 98.88251941078296, 'easy': 99.97387783534329}
{'epoch': 4, 'train_loss': 0.026165247616384914, 'val_loss': 0.050065040588378906, 'found': 78.29360100376411, 'case1': 98.66482838690952, 'easy': 99.96517044712438}
finishing seed:0 -> best: {'epoch': 1, 'train_loss': 0.03485945930802605, 'val_loss': 0.05212624371051788, 'found': 79.92471769134254, 'case1': 98.38183005587403, 'easy': 99.9651

## 14. Results per slice
**Why:** the headline question is whether alignment fixes case2s.
**Does:** val per slice against the word2vec baseline and the Step 3 siamese model; test once, at the end, for the best configuration; row added to `results.md`.
**Output:** slice table

### 14a. Val, per seed (comparison with earlier models; no run is picked, see 14b)
- Same loop as 14b on `val_loader` / `val` / `val_label`; keep the predictions in `val_res[seed]`
- One table, rows = slices, one column per seed (`set_index("slice")["% correct"]`, then `pd.DataFrame`)
- Next to it: the word2vec baseline and the `case2s`-trained siamese val numbers (`results.md`, in brackets)

In [29]:
# 14a: val per seed
res_val = {}
for seed in [0, 1, 2]:
    print(f"{'='*20} seed:{seed} {'='*20}")
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt"))

    model = ESIM()
    model.load_state_dict(ckpt["state_dict"])
    # res = evaluate(model, test_loader, ot_test, ot_test_label)
    # print(res)
    with torch.no_grad():
        val_preds = (predict(model, val_loader) >= THRESHOLD).int().numpy()
        res_val[seed] = val_preds
        np.save(os.path.join(CHECKPOINT_DIR, f"preds_val_seed_{seed}.npy"), val_preds)   # read by 14d
        val_slices = review_typical_slices(val, val_preds, val_label)
        print(val_slices)

==================== seed:0 ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3384       6     99.8%
1                  positives: non-identical    797      628     169     78.8%
2     neg: case1 (same street, diff number)  13781    13543     238     98.3%
3  neg: case2 (same number, similar street)     34       15      19     44.1%
4                         neg: case3 (easy)  22969    22963       6    100.0%
5   neg: case2s (swapped look-alike street)    894      650     244     72.7%
6                        overall (accuracy)  41865    41183     682     98.4%
==================== seed:1 ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3349      41     98.8%
1                  positives: non-identical    797      640     157     80.3%
2     neg: case1 (same street, diff number) 

### 14b. Test, all 3 seeds
All three seeds were scored on test (2026-11-05) before a run was chosen, so no single run is picked: test is reported as the min–max range over the seeds. Nothing is chosen from these numbers.

In [30]:
res_test = {}
for seed in [0, 1, 2]:
    print(f"{'='*20} seed:{seed} {'='*20}")
    ckpt = torch.load(os.path.join(CHECKPOINT_DIR, f"esim_seed_{seed}.pt"))

    model = ESIM()
    model.load_state_dict(ckpt["state_dict"])
    # res = evaluate(model, test_loader, ot_test, ot_test_label)
    # print(res)
    with torch.no_grad():
        test_preds = (predict(model, test_loader) >= THRESHOLD).int().numpy()
        res_test[seed] = test_preds
        np.save(os.path.join(CHECKPOINT_DIR, f"preds_test_seed_{seed}.npy"), test_preds)   # read by 14d
        test_slices = review_typical_slices(test, test_preds, test_label)
        print(test_slices)

==================== seed:0 ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3526      17     99.5%
1                  positives: non-identical    843      646     197     76.6%
2     neg: case1 (same street, diff number)  14448    14064     384     97.3%
3  neg: case2 (same number, similar street)     15        4      11     26.7%
4                         neg: case3 (easy)  23942    23915      27     99.9%
5   neg: case2s (swapped look-alike street)   1053      759     294     72.1%
6                        overall (accuracy)  43844    42914     930     97.9%
==================== seed:1 ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3543     3430     113     96.8%
1                  positives: non-identical    843      661     182     78.4%
2     neg: case1 (same street, diff number) 

### 14c. `results.md` row
- Each slice as test min–max (val min–max), same columns as the `case2s` table

### 14d. `min_count` 2 vs 1, per slice and seed
Reads the predictions 14a / 14b saved for each setting (no models reloaded). Needs one run of the notebook per setting: `W2V_MIN_COUNT = 2` (training is skipped, the runs exist; saves `esim_mc2/preds_*`), then `W2V_MIN_COUNT = 1` (trains, saves `esim_mc1/preds_*`). D2–D4 define their group with the `min_count=2` vocab, so rerunning them under each setting compares the same rows.


In [31]:
# 14d: % correct per slice, mc2 vs mc1, one column per (setting, seed)  (set up by Claude 2026-11-05)
def slice_table(split_df, labels, split):
    cols = {}
    for mc in [2, 1]:
        for seed in [0, 1, 2]:
            path = f"checkpoints/esim_mc{mc}/preds_{split}_seed_{seed}.npy"
            if os.path.exists(path):
                cols[(f"mc{mc}", seed)] = review_typical_slices(split_df, np.load(path), labels).set_index("slice")["% correct"]
    return pd.DataFrame(cols)

for name, split_df, labels in [("val", val, val_label), ("test", test, test_label)]:
    print(f"{'='*20} {name} {'='*20}")
    display(slice_table(split_df, labels, name))


==================== val ====================


mc1                
                                               0       1       2
slice                                                           
positives: identical                       99.8%   98.8%  100.0%
positives: non-identical                   78.8%   80.3%   81.1%
neg: case1 (same street, diff number)      98.3%   98.7%   98.1%
neg: case2 (same number, similar street)   44.1%   52.9%   50.0%
neg: case3 (easy)                         100.0%  100.0%  100.0%
neg: case2s (swapped look-alike street)    72.7%   83.6%   74.4%
overall (accuracy)                         98.4%   98.7%   98.4%

==================== test ====================


mc1               
                                              0      1       2
slice                                                         
positives: identical                      99.5%  96.8%  100.0%
positives: non-identical                  76.6%  78.4%   78.4%
neg: case1 (same street, diff number)     97.3%  98.3%   95.7%
neg: case2 (same number, similar street)  26.7%  26.7%   33.3%
neg: case3 (easy)                         99.9%  99.9%   99.9%
neg: case2s (swapped look-alike street)   72.1%  79.6%   74.5%
overall (accuracy)                        97.9%  98.2%   97.5%

## Diagnostics
One-off inspections, kept out of the pipeline cells above. Results are written into the markdown (question → result → query).

### D1. Why do seeds 0 and 1 reject ~150 identical positives? (test, seed 0, 10-row samples)
**Hypothesis:** rare house numbers are not in `vocab` → `UNK`. A case1 negative with two rare numbers (`4817 MAIN ST` / `4819 MAIN ST`) becomes `[UNK, MAIN, ST]` vs `[UNK, MAIN, ST]`, label 0, so the model sees the same input as an identical positive with a rare number, with the opposite label.

**Result (sample):** all 10 misses have an unknown number (`None`), all 4-digit; the 10 hits all have a known number.

In [32]:
# Group = numbers that were UNK in the original run, fixed across settings so mc1 vs mc2 compare the same rows
group_vocab = Word2Vec.load("checkpoints/word2vec_50_mc2.model").wv.key_to_index
for p in [0, 1]:
    preds = res_test[0]   # pick the seed here; the mask and the table must use the same predictions
    mask = (test["address_a"] == test["address_b"]).values & (test_label == 1).values & (preds == p)

    df = find_random_sample_given_mask(test, preds, test_label, mask)

    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])

    display(df[["address_a", "address_b", "predicted", "label", "address_a_ids", "address_b_ids"]])

,address_a,address_b,predicted,label,address_a_ids,address_b_ids
766,3015 MAIN ST,3015 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
3716,2551 BERLIN TPKE,2551 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3756,3123 BERLIN TPKE,3123 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3771,3075 BERLIN TPKE,3075 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3836,3206 BERLIN TPKE,3206 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3139,1414 BLUE HILLS AVE,1414 BLUE HILLS AVE,0,1,"[None, 479, 133, 2]","[None, 479, 133, 2]"
1982,2252 MAIN ST,2252 MAIN ST,0,1,"[None, 3, 0]","[None, 3, 0]"
3837,3210 BERLIN TPKE,3210 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3816,3375 BERLIN TPKE,3375 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"
3752,3111 BERLIN TPKE,3111 BERLIN TPKE,0,1,"[None, 46, 7]","[None, 46, 7]"


,address_a,address_b,predicted,label,address_a_ids,address_b_ids
472,7 KENT RD,7 KENT RD,1,1,"[36, 269, 1]","[36, 269, 1]"
1493,106 WALL ST,106 WALL ST,1,1,"[350, 217, 0]","[350, 217, 0]"
703,319 BARNUM AVE,319 BARNUM AVE,1,1,"[801, 153, 2]","[801, 153, 2]"
4247,725 JARVIS ST,725 JARVIS ST,1,1,"[654, None, 0]","[654, None, 0]"
575,8 DODD RD,8 DODD RD,1,1,"[95, None, 1]","[95, None, 1]"
53,7 WAKELEY ST,7 WAKELEY ST,1,1,"[36, 2990, 0]","[36, 2990, 0]"
2231,2500 MAIN ST,2500 MAIN ST,1,1,"[2517, 3, 0]","[2517, 3, 0]"
1356,914 HARTFORD TPKE,914 HARTFORD TPKE,1,1,"[1264, 21, 7]","[1264, 21, 7]"
656,574 STILLMAN ST,574 STILLMAN ST,1,1,"[827, 1254, 0]","[827, 1254, 0]"
1025,1019 HOPE ST,1019 HOPE ST,1,1,"[1800, 195, 0]","[1800, 195, 0]"


### D2. Is the `UNK` number the pattern across the whole slice? (val, all identical positives)
**Group:** both strings start with a house number, both numbers are unknown to `vocab` (`ids[0] is None`), the rest of the address is identical (`compare_none_numbers`).

**Notes (David):**
- We isolate the DataFrame, for all examples where the only differentatior, is that they BOTH START WITH a house_number and for both, `Embeddings[0] == None`
- After testing, we figure out that this exact case is why we failed, accounting for 99.9% of failures on `positives:identical`
- It has a hard time comparing two numbers, whose house_number's embedding is `<unk>`

**Result (val):** every miss is in this group (seed 0: 49 of 49, seed 1: 69 of 70); with a known number the model almost never rejects an identical pair.

| identical positives, found | seed 0 | seed 1 | seed 2 |
|---|---|---|---|
| both numbers `UNK` (n = 78) | 29 | 9 | 78 |
| known number (n = 3,312) | 3,312 | 3,311 | 3,312 |


In [33]:
# D2: UNK-number share, missed vs found identical positives
extract_non_number = lambda x: x.str.replace(r"^(\d+\s+)", "", regex= True)   # defined before the loop uses it
for seed in [0, 1, 2]:
    preds = res_val[seed]   # pick the seed here; the mask and the table must use the same predictions

    
    # mask = (val["address_a"] == val["address_b"]).values & (val_label == 1).values & (preds == p)
    mask = (val["address_a"] == val["address_b"]).values

    df = find_random_sample_given_mask(val, preds, val_label, mask, size = None)

    
    # Add Tokenized List 
    for check in ["a", "b"]:
        #Embedded Tokens
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        #Starts With Number?
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])

    
    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) & 
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) & 
        (df["non_number_a"] == df["non_number_b"])
    )
    display_list = [
        "address_a", "address_b", 
        "predicted", "label", 
        "address_a_ids", "address_b_ids", 
        "address_a_num_start", "address_b_num_start", 
        "non_number_a","non_number_b", 
        "compare_none_numbers"]

    # display(df[display_list])
    print(f"{'='*20} seed:{seed} {'='*20}")
    display(df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False))

==================== seed:0 ====================


predicted,0,1
compare_none_numbers,,
True,6.0,72.0
False,NaN,3312.0


==================== seed:1 ====================


predicted,0,1
compare_none_numbers,,
True,40,38
False,1,3311


==================== seed:2 ====================


predicted,1
compare_none_numbers,
True,78
False,3312


### D3. Does seed 2 pay for its 100% on case1? (val, case1 negatives)
Same group as D2, on case1 rows: `4817 MAIN ST` / `4819 MAIN ST` with both numbers `UNK` is the same input as an identical pair with a rare number.

**Notes (David):**
Since we've concluded that the model struggles to compare addresses when house_number except `<unk>` (other than seed 2), we would expect this to carry over for negatives as well
- As shown, `seed0` and `seed1` seem to be able differentiate quite well.
- In comparison `seed2` categorizes all negative values as 

**Result (val):** yes. Seed 2 predicts "match" on all 161 such case1 rows (all wrong). Seeds 0/1 reject most of them, but they also reject most of the 78 identical `UNK` pairs in D2: no seed tells the two apart (the input is the same), each picks one default for the whole group. Seeds 0/1: "no match"; seed 2: "match".

| case1, rejected | seed 0 | seed 1 | seed 2 |
|---|---|---|---|
| both numbers `UNK` (n = 161) | 141 | 158 | 0 |
| known numbers (n = 13,620) | 13,404 | 13,496 | 13,528 |

Outside the `UNK` group seed 2 has the fewest case1 errors (92 vs 216 / 124).


In [34]:
for seed in [0, 1, 2]:
    preds = res_val[seed]
    mask = (val["neg_type"] == "case1").values & (val_label == 0).values   # D2 had: identical strings

    df = find_random_sample_given_mask(val, preds, val_label, mask, size = None)

    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])

    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) &
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) &
        (df["non_number_a"] == df["non_number_b"])
    )

    print(f"{'='*20} seed:{seed} {'='*20}")
    display(df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False))


==================== seed:0 ====================


predicted,0,1
compare_none_numbers,,
True,21,140
False,13522,98


==================== seed:1 ====================


predicted,0,1
compare_none_numbers,,
True,111,50
False,13495,125


==================== seed:2 ====================


predicted,0,1
compare_none_numbers,,
True,12,149
False,13504,116


### D4. D2 + D3 on test (confirmation only; nothing is chosen from it)
- Same two row groups, same `compare_none_numbers` column, test predictions `res_test`
- **Result (test):** same pattern as val, larger group. All identical-pair misses of seeds 0/1 are in the `UNK` group (146 of 146, 158 of 158); 404 of seed 2's 566 case1 errors are in it (the other 162 = fewest outside the group, vs 225 / 207).

| both numbers `UNK`, same street | seed 0 | seed 1 | seed 2 |
|---|---|---|---|
| identical positives found (n = 178) | 32 | 20 | 178 |
| case1 negatives rejected (n = 404) | 381 | 387 | 0 |

**Conclusion (D1–D4):** with frozen word-level embeddings and `min_count=2`, every rare house number becomes the same `UNK`. A case1 pair with two rare numbers on the same street is then the same input as an identical pair with a rare number, but with the opposite label. The model can only pick one answer for that input: seeds 0/1 lose identical pairs, seed 2 loses case1. Training or seeds can't fix it; the house number has to reach the model (e.g. the char expansion in the spec). This explains the test gap in 14b (identical 95.5–95.9 vs 100; case1 96.1 vs ~98.3), larger on test because test has more such rows (404 vs 161 case1).


In [35]:
# D4: D2 (identical positives) and D3 (case1 negatives) on test (written by Claude 2026-11-05)
# Needs extract_non_number from D2 and res_test from 14b.
def unk_number_table(df_split, labels, preds, mask):
    df = find_random_sample_given_mask(df_split, preds, labels, mask, size = None)
    for check in ["a", "b"]:
        df[f"address_{check}_ids"] = df[f"address_{check}"].apply(lambda x: [group_vocab.get(word) for word in tokenize(x)])
        df[f"address_{check}_num_start"] = df[f"address_{check}"].str.contains(r"^\d+")
        df[f"non_number_{check}"] = extract_non_number(df[f"address_{check}"])
    df["compare_none_numbers"] = (
        (df["address_a_ids"].str[0].isna()) &
        (df["address_b_ids"].str[0].isna()) &
        (df["address_a_num_start"] == True) &
        (df["address_b_num_start"] == True) &
        (df["non_number_a"] == df["non_number_b"])
    )
    return df.groupby(["compare_none_numbers", "predicted"])["predicted"].size().unstack("predicted").sort_index(ascending = False)

row_groups = {
    "D2 identical positives (1 = correct)": (test["address_a"] == test["address_b"]).values & (test_label == 1).values,
    "D3 case1 negatives (0 = correct)":     (test["neg_type"] == "case1").values & (test_label == 0).values,
}
for name, mask in row_groups.items():
    for seed in [0, 1, 2]:
        print(f"{'='*10} {name} | seed:{seed} {'='*10}")
        display(unk_number_table(test, test_label, res_test[seed], mask))


========== D2 identical positives (1 = correct) | seed:0 ==========


predicted,0,1
compare_none_numbers,,
True,16,162
False,1,3364


========== D2 identical positives (1 = correct) | seed:1 ==========


predicted,0,1
compare_none_numbers,,
True,106,72
False,7,3358


========== D2 identical positives (1 = correct) | seed:2 ==========


predicted,1
compare_none_numbers,
True,178
False,3365


... (2 outputs left out)

========== D3 case1 negatives (0 = correct) | seed:1 ==========


predicted,0,1
compare_none_numbers,,
True,316,88
False,13884,160


========== D3 case1 negatives (0 = correct) | seed:2 ==========


predicted,0,1
compare_none_numbers,,
True,64,340
False,13768,276
